In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Add project root to sys.path (so imports like src.data work)
project_root = Path('/home/swermuth/pmof-code')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import logger
from src import DATA_BASE_DIR as data_base_dir

from ultralytics import YOLO

from src.visualization import results_to_frames, save_video, VIZ_PARAMS

In [3]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
import shutil

from src import DATA_BASE_DIR as data_base_dir
from src.data import list_record_ids
record_ids = list_record_ids()
from src.visualization import VIZ_PARAMS
bbox_colors = VIZ_PARAMS['gt_bbox_colors']

from src.data import imgid_to_imgpath, imgid_to_annpath, recid_to_annpath, recordid_to_imageids, read_annotation, list_record_ids

In [ ]:
record_ids = list_record_ids()
record_ids

In [4]:
train_record_ids = ["rec4", "rec22", "rec25", "rec27", "rec28"]
val_record_ids = ["rec29", "rec30"]

In [5]:
from src.data import imgid_to_imgpath, imgid_to_annpath, recid_to_annpath
from src.data import recordid_to_imageids, read_annotation

# Restructure
Doc for Dataset Structure:  https://docs.ultralytics.com/datasets/classify#folder-structure-example

In [8]:
#make directories
PMOF_actioncls_base_dir = Path('/home/swermuth/PMOF_actioncls/')

for split in ["train", "val"]:
    for cls in ["seated", "other"]:
        (PMOF_actioncls_base_dir / split / cls).mkdir(parents=True, exist_ok=True)

In [9]:
def copy_images_by_action(
    record_ids,
    output_dir,
    split,
    seated_action="seated",
):
    """
    Copy images into:
        output_dir/
        ├── {split}/
        │   ├── seated/
        │   └── other/

    An image is classified as 'seated' only if all annotations
    have action == seated_action. If it contains any other action,
    it is classified as 'other'.

    Returns:
        Number of images processed.
    """
    output_dir = Path(output_dir)

    seated_dir = output_dir / split / "seated"
    other_dir = output_dir / split / "other"

    seated_dir.mkdir(parents=True, exist_ok=True)
    other_dir.mkdir(parents=True, exist_ok=True)

    images_counter = 0

    for rec_id in record_ids:
        image_ids = recordid_to_imageids(rec_id)

        for image_id in image_ids:
            images_counter += 1

            # Get image and annotation paths
            imgpath = Path(imgid_to_imgpath(image_id))
            annpath = imgid_to_annpath(image_id)

            # Sanity check
            if not imgpath.is_file():
                print(f"Warning: image not found: {imgpath}")
                continue

            anns = read_annotation(annpath, image_id)

            # Only person annotations matter for action classification
            person_anns = [
                ann for ann in anns
                if ann.category_name == "person"
            ]

            # Any non-seated action -> "other"
            has_other_action = any(
                box.action != seated_action
                for box in person_anns
            )

            destination_dir = (
                other_dir if has_other_action else seated_dir
            )

            destination = destination_dir / imgpath.name

            shutil.copy2(imgpath, destination)

    print(f"{split}: copied {images_counter} images")

    return images_counter

In [10]:
copy_images_by_action(val_record_ids, PMOF_actioncls_base_dir, 'val')

val: copied 1359 images


1359

In [11]:
copy_images_by_action(train_record_ids, PMOF_actioncls_base_dir, 'train')

train: copied 3351 images


3351

# Train Model

In [17]:
from ultralytics import YOLO

# Load a model
#model = YOLO("yolo26n-cls.yaml")  # build a new model from YAML
model = YOLO("yolo26s-cls.pt")  # load a pretrained model (recommended for training)
#model = YOLO("yolo26n-cls.yaml").load("yolo26n-cls.pt")  # build from YAML and transfer weights

# Train the model
results = model.train(data="/home/swermuth/PMOF_actioncls/", epochs=10, imgsz=320)

Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/swermuth/PMOF_actioncls/, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

curl: (7) Couldn't connect to server

curl: (7) Couldn't connect to server

curl: (7) Couldn't connect to server



train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 71.8±6.8 MB/s, size: 2871.5 KB)
train: Scanning /home/swermuth/PMOF_actioncls/train... 3351 images, 0 corrupt: 100% ━━━━━━━━━━━━ 3351/3351 203.9it/s 16.4s0.2s
train: New cache created: /home/swermuth/PMOF_actioncls/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 62.7±5.7 MB/s, size: 2603.2 KB)
val: Scanning /home/swermuth/PMOF_actioncls/val... 1359 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1359/1359 222.1it/s 6.1s0.1s
val: New cache created: /home/swermuth/PMOF_actioncls/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001667, momentum=0.9) with parameter groups 39 weight(decay=0.0), 40 weight(decay=0.0005), 40 bias(decay=0.0)
Image sizes 320 train, 320 val
Using 8 dataloader workers
Logging results to /home/swermuth/pmof-symposium/runs/classify/train
Starting training for 10 epochs...

      Epo